# Agentic Memory: A Complete Tutorial

**How to give AI agents the ability to remember — types, architecture, and working implementations (using OpenAI models).**

Large Language Models are *stateless*: every API call starts from a blank slate. An "agent" that forgets the user's name between sessions, repeats mistakes it already made, and re-derives the same facts over and over is not much of an agent. **Agentic memory** is the set of design patterns that gives an LLM-powered agent persistent, useful state.

### What you'll build in this notebook

1. **Why memory?** — the statelessness problem, demonstrated
2. **Memory taxonomy** — short-term / working memory vs. long-term (episodic, semantic, procedural)
3. **Architecture** — how the pieces fit together in a real agent
4. **Short-term memory** — conversation history management
5. **Context-window management** — token counting, sliding windows, summarization
6. **Semantic memory** — extracting and storing durable facts with structured outputs
7. **Episodic memory** — remembering past sessions as retrievable "episodes"
8. **Procedural memory** — learning rules and behaviors from feedback
9. **Retrieval scoring** — recency + importance + relevance (the *Generative Agents* formula)
10. **Agent-driven memory** — a memory *tool* the model uses via function calling
11. **Putting it all together** — a `MemoryAgent` that combines every layer
12. **Advanced topics** — forgetting, consolidation, vector stores, caching, hosted state
13. **Best practices & pitfalls**

> **Requirements:** Python 3.10+, an OpenAI API key in the `OPENAI_API_KEY` environment variable. Every code cell is runnable end-to-end.

## 0. Setup

In [1]:
%pip install -q openai pydantic tiktoken

/Users/aadhilimam/Desktop/my_code/articles/Agentic Memory/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
import math
import time
import textwrap
from datetime import datetime, timezone
from pathlib import Path

from openai import OpenAI

MODEL = "gpt-4o"  # swap for "gpt-4.1", "gpt-5", etc. as you prefer

# Reads OPENAI_API_KEY from the environment
client = OpenAI()


def llm(messages: list[dict], system: str | None = None, max_tokens: int = 1024) -> str:
    """Small helper: one chat-completions call, returns the reply text."""
    msgs = ([{"role": "system", "content": system}] if system else []) + messages
    response = client.chat.completions.create(
        model=MODEL,
        max_completion_tokens=max_tokens,
        messages=msgs,
    )
    return response.choices[0].message.content


print("Client ready. Model:", MODEL)

Client ready. Model: gpt-4o


## 1. Why memory? The statelessness problem

The Chat Completions API has **no server-side conversation state**. Each request is independent — the model only "knows" what is in the current request's `messages` array.

Watch what happens when we make two *separate* calls:

In [3]:
# Call 1: introduce ourselves
print("Call 1:", llm([{"role": "user", "content": "Hi! My name is Aadhil and my favorite language is Python."}], max_tokens=256))

# Call 2: a brand-new request — the model has no idea who we are
print("\nCall 2:", llm([{"role": "user", "content": "What's my name and favorite language?"}], max_tokens=256))

Call 1: Hello Aadhil! It's great to meet you. Python is a wonderful language with a lot of versatility, whether you're into data science, web development, automation, or any other field. Is there anything specific you enjoy doing with Python, or any projects you're currently working on?

Call 2: I'm sorry, but I don't have access to personal data about individuals unless it has been shared with me in the course of our conversation. Since I have no information about you in this session, I can't determine your name or favorite language. If you'd like to share more about yourself, feel free to do so!


The second call can't answer — the model genuinely doesn't know. **Everything an agent "remembers" is something your application put back into the prompt.** Agentic memory is the engineering discipline of deciding *what* to store, *where* to store it, *when* to write it, and *how* to get the right pieces back into context at the right time.

## 2. Memory taxonomy

Agent memory design borrows heavily from cognitive science. The standard taxonomy:

```
                        AGENT MEMORY
                             │
            ┌────────────────┴────────────────┐
            │                                 │
     SHORT-TERM MEMORY                 LONG-TERM MEMORY
     (working memory)                  (persistent stores)
            │                                 │
   ┌────────┴────────┐          ┌─────────────┼──────────────┐
   │                 │          │             │              │
  Conversation   Scratchpad  EPISODIC      SEMANTIC      PROCEDURAL
  history        (current    "what          "what           "how
  (this turn's   reasoning,   happened"      is true"        to act"
  context)       tool state)
```

| Type | Human analogy | Agent implementation | Example |
|---|---|---|---|
| **Working / short-term** | What you're holding in mind right now | The `messages` array in the current request (the context window) | The last 20 turns of this chat |
| **Episodic** | Remembering specific past events | Stored summaries of past sessions/interactions, retrievable by similarity | "Last Tuesday we debugged the auth bug together" |
| **Semantic** | Facts and general knowledge | A store of extracted facts about the user, project, world | "User prefers tabs; deploys on Fridays; timezone is IST" |
| **Procedural** | Skills — riding a bike, muscle memory | Learned rules/instructions injected into the system prompt; updated tool-use policies | "Always run tests before claiming a fix works" |

Two axes cut across this taxonomy and drive every design decision:

1. **Scope** — *in-session* (dies with the conversation) vs. *cross-session* (survives restarts).
2. **Who writes it** — *the application* (deterministic pipelines: summarizers, extractors) vs. *the agent itself* (the model decides what's worth remembering via a memory tool).

The strongest agents use **both**: deterministic pipelines for reliability, agent-driven writes for judgment.

## 3. Architecture of a memory-augmented agent

A production memory system has four moving parts around the LLM call:

```
                 ┌──────────────────────────────────────────────┐
                 │                MEMORY MANAGER                │
                 │                                              │
 user input ───▶│  1. RETRIEVE   pull relevant memories        │
                 │       │        (semantic + episodic +        │
                 │       │         procedural rules)            │
                 │       ▼                                      │
                 │  2. ASSEMBLE   build the prompt:             │
                 │       │        system = persona + rules      │
                 │       │                 + recalled memories  │
                 │       │        messages = working memory     │
                 │       ▼                                      │
                 │  ┌─────────┐                                 │
                 │  │   LLM   │──── may call a memory TOOL ────▶│──┐
                 │  └────┬────┘     (read/write long-term)      │  │
                 │       ▼                                      │  │
                 │  3. RESPOND   answer goes to the user        │  │
                 │       │                                      │  │
                 │       ▼                                      │  │
                 │  4. CONSOLIDATE (write policy)               │  │
                 │       - append turn to working memory        │  │
                 │       - extract new facts → semantic store   │  │
                 │       - on session end → episodic store      │  │
                 │       - on feedback → update procedural      │  │
                 └──────────────────────────────────────────────┘  │
                                     ▲                             │
                                     │                             │
                   ┌─────────────────┴───────────────┐             │
                   │         LONG-TERM STORES        │◀────────────┘
                   │  semantic.json  episodes.json   │
                   │  rules.json     /memories/*.md  │
                   └─────────────────────────────────┘
```

The rest of this notebook implements each box, then assembles them into one agent.

> **Key insight:** memory is a *prompt-assembly problem*, not a database problem. The hard parts are the **write policy** (what's worth keeping) and the **read policy** (what's worth injecting into a limited context window) — the storage itself can be a JSON file.

## 4. Short-term (working) memory

The simplest memory is the conversation history itself. Since the API is stateless, the application must accumulate turns and resend them:

In [4]:
class ConversationMemory:
    """Working memory: accumulates the current conversation's turns."""

    def __init__(self, system: str = "You are a helpful assistant."):
        self.system = system
        self.messages: list[dict] = []

    def send(self, user_message: str, max_tokens: int = 1024) -> str:
        self.messages.append({"role": "user", "content": user_message})
        reply = llm(self.messages, system=self.system, max_tokens=max_tokens)
        # Append the assistant turn so the next call sees it
        self.messages.append({"role": "assistant", "content": reply})
        return reply


chat = ConversationMemory()
print(chat.send("My name is Aadhil and my favorite language is Python."))
print("---")
print(chat.send("What's my name and favorite language?"))  # now it remembers

Hi Aadhil! It's great to meet you. Python is a fantastic programming language—it's versatile, user-friendly, and widely used in many fields. Do you have any specific projects you're working on, or is there anything you'd like to know more about Python?
---
Your name is Aadhil, and your favorite language is Python.


This works — until the conversation outgrows the context window (or your budget). Working memory needs *management*.

## 5. Managing the context window

Three standard strategies, in increasing sophistication:

| Strategy | Idea | Loses information? |
|---|---|---|
| **Sliding window** | Keep only the last *N* turns / *T* tokens | Yes — silently drops old turns |
| **Summarization (compaction)** | Replace old turns with an LLM-written summary | Compresses; keeps the gist |
| **Server-side state** (Responses API) | Let OpenAI store the thread; chain by `previous_response_id` | Managed; see §5.4 |

### 5.1 Counting tokens

Any budget-based strategy starts with token counts. For OpenAI models, `tiktoken` is the official tokenizer (GPT-4o and newer use the `o200k_base` encoding). Note that chat formatting adds a few tokens of overhead per message, so treat counts as close estimates:

In [5]:
import tiktoken

try:
    ENCODING = tiktoken.encoding_for_model(MODEL)
except KeyError:
    ENCODING = tiktoken.get_encoding("o200k_base")  # fallback for models tiktoken doesn't know yet


def count_tokens(messages: list[dict], system: str | None = None) -> int:
    """Approximate prompt size: content tokens + ~4 tokens/message chat overhead."""
    total = len(ENCODING.encode(system)) + 4 if system else 0
    for m in messages:
        if isinstance(m["content"], str):
            total += len(ENCODING.encode(m["content"])) + 4
    return total


print("Current conversation size:", count_tokens(chat.messages, chat.system), "tokens")

Current conversation size: 113 tokens


### 5.2 Sliding window

Trim from the *front* (oldest turns) until the conversation fits a budget. Always trim in whole user→assistant pairs so the history still starts with a `user` message:

In [6]:
def sliding_window(messages: list[dict], budget_tokens: int, system: str | None = None) -> list[dict]:
    """Drop oldest turn-pairs until the conversation fits the token budget."""
    trimmed = list(messages)
    while len(trimmed) > 2 and count_tokens(trimmed, system) > budget_tokens:
        trimmed = trimmed[2:]  # drop the oldest user+assistant pair
    return trimmed


demo = [
    {"role": "user", "content": "Tell me about the history of Rome. " * 40},
    {"role": "assistant", "content": "Rome was founded... " * 60},
    {"role": "user", "content": "And Greece?"},
    {"role": "assistant", "content": "Greece has an equally rich history..."},
    {"role": "user", "content": "Compare the two briefly."},
]

print("Before:", len(demo), "messages,", count_tokens(demo), "tokens")
windowed = sliding_window(demo, budget_tokens=300)
print("After: ", len(windowed), "messages,", count_tokens(windowed), "tokens")

Before: 5 messages, 598 tokens
After:  3 messages, 28 tokens


### 5.3 Summarization (compaction)

The sliding window *forgets* — the user's name from turn 1 is gone. Better: when the history gets long, replace old turns with an LLM-written summary and keep the recent turns verbatim. The summary becomes a compressed piece of working memory:

In [7]:
def compact(messages: list[dict], keep_recent: int = 4) -> list[dict]:
    """Summarize everything except the last `keep_recent` messages."""
    if len(messages) <= keep_recent:
        return messages

    old, recent = messages[:-keep_recent], messages[-keep_recent:]
    transcript = "\n".join(f"{m['role'].upper()}: {m['content']}" for m in old)

    summary = llm(
        [{"role": "user", "content": transcript}],
        system=(
            "You compress conversation history for an AI agent's context window. "
            "Write a dense summary that preserves: user identity and preferences, "
            "decisions made, open questions, and any facts the agent will need later. "
            "Omit pleasantries."
        ),
    )

    # The summary re-enters the history as context inside the first user turn
    return [
        {"role": "user", "content": f"[Summary of the conversation so far]\n{summary}"},
        {"role": "assistant", "content": "Understood — I have the context. Continuing."},
        *recent,
    ]


compacted = compact(chat.messages + demo, keep_recent=2)
print(f"{len(chat.messages) + len(demo)} messages -> {len(compacted)} messages\n")
print("Summary turn:\n", textwrap.fill(compacted[0]["content"], 100))

9 messages -> 4 messages

Summary turn:
 [Summary of the conversation so far] Aadhil is a user whose favorite programming language is Python.
He requested information about the history of Rome, resulting in a repetitive query. He also asked
about the history of Greece, an open question pending further clarification.


### 5.4 Server-side conversation state (Responses API)

OpenAI's **Responses API** can hold conversation state for you: pass `store=True` and chain follow-up turns with `previous_response_id` instead of resending the whole history. With `truncation="auto"`, the server drops middle context automatically when the thread approaches the window limit.

```python
first = client.responses.create(
    model=MODEL,
    input="My name is Aadhil and my favorite language is Python.",
    store=True,
)

followup = client.responses.create(
    model=MODEL,
    previous_response_id=first.id,   # server recalls the earlier turn
    input="What's my name?",
    truncation="auto",
)
print(followup.output_text)
```

This is convenient, but note what it is — **hosted working memory**, not long-term memory. It's still one conversation thread; it doesn't extract facts, learn rules, or recall across threads. The rest of this notebook builds those layers.

## 6. Semantic memory: durable facts

Semantic memory stores **what is true** — facts about the user, the project, the world — independent of when you learned them. The pipeline:

1. **Extract** candidate facts from a conversation (an LLM call with structured output).
2. **Deduplicate / reconcile** against the existing store.
3. **Persist** (here: a JSON file; in production: a database).
4. **Recall** relevant facts into the system prompt of future sessions.

We use `client.chat.completions.parse` with a Pydantic schema so extraction is guaranteed-valid JSON:

In [8]:
from pydantic import BaseModel


class Fact(BaseModel):
    fact: str          # one atomic statement, e.g. "User's timezone is IST"
    category: str      # "identity" | "preference" | "project" | "constraint" | "other"


class FactList(BaseModel):
    facts: list[Fact]


def parse_llm(user_content: str, system: str, schema, max_tokens: int = 1024):
    """Structured-output helper: returns a validated Pydantic instance."""
    completion = client.chat.completions.parse(
        model=MODEL,
        max_completion_tokens=max_tokens,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_content},
        ],
        response_format=schema,
    )
    return completion.choices[0].message.parsed


def extract_facts(transcript: str) -> list[Fact]:
    """Ask the model to pull out durable, atomic facts worth remembering long-term."""
    result = parse_llm(
        transcript,
        system=(
            "Extract durable facts about the user or their work that would be useful "
            "in FUTURE conversations. One atomic fact per entry. "
            "Skip small talk, one-off requests, and anything session-specific. "
            "Return an empty list if nothing qualifies."
        ),
        schema=FactList,
    )
    return result.facts


sample_transcript = """
USER: Hey, I'm Aadhil — I'm a data engineer in Colombo working on a legal CRM side project.
ASSISTANT: Nice to meet you! How can I help with the CRM?
USER: I'm using React with Tailwind. Also FYI I prefer code examples without semicolons.
ASSISTANT: Got it — React + Tailwind, no semicolons.
USER: What's the weather like today?
ASSISTANT: I don't have live weather access, sorry!
"""

facts = extract_facts(sample_transcript)
for f in facts:
    print(f"[{f.category:11s}] {f.fact}")

[User Profession] Aadhil is a data engineer.
[User Project] Aadhil is working on a legal CRM side project.
[User Location] Aadhil is based in Colombo.
[User Technology Preference] Aadhil is using React with Tailwind for his project.
[User Coding Style] Aadhil prefers code examples without semicolons.


Note what the extractor *skipped*: the weather question. A good **write policy** is as much about what you don't store as what you do.

Now the store itself — persistence plus dedup on write:

In [9]:
class SemanticMemory:
    """A persistent store of atomic facts, with naive dedup on write."""

    def __init__(self, path: str = "./memory_data/semantic.json"):
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.facts: list[dict] = json.loads(self.path.read_text()) if self.path.exists() else []

    def _save(self):
        self.path.write_text(json.dumps(self.facts, indent=2))

    def add(self, new_facts: list[Fact]):
        existing = {f["fact"].lower() for f in self.facts}
        for f in new_facts:
            if f.fact.lower() not in existing:
                self.facts.append({
                    "fact": f.fact,
                    "category": f.category,
                    "created_at": datetime.now(timezone.utc).isoformat(),
                })
        self._save()

    def recall(self, query: str | None = None, limit: int = 10) -> list[str]:
        """Return facts, optionally ranked by keyword overlap with a query."""
        if not query:
            return [f["fact"] for f in self.facts[:limit]]
        q_words = set(query.lower().split())
        scored = sorted(
            self.facts,
            key=lambda f: len(q_words & set(f["fact"].lower().split())),
            reverse=True,
        )
        return [f["fact"] for f in scored[:limit]]

    def as_system_block(self, query: str | None = None) -> str:
        recalled = self.recall(query)
        if not recalled:
            return ""
        return "Known facts about the user:\n" + "\n".join(f"- {fact}" for fact in recalled)


semantic = SemanticMemory()
semantic.add(facts)
print(semantic.as_system_block())

Known facts about the user:
- Aadhil is a data engineer.
- Aadhil is working on a legal CRM side project.
- Aadhil is based in Colombo.
- Aadhil is using React with Tailwind for his project.
- Aadhil prefers code examples without semicolons.


In [10]:
# The payoff: a BRAND-NEW session (empty history) that still knows the user.
print(llm(
    [{"role": "user", "content": "Suggest a component library for my project. Show a tiny code snippet in my preferred style."}],
    system="You are a helpful assistant.\n\n" + semantic.as_system_block(),
    max_tokens=512,
))

For your project, you might find the Headless UI library very useful. It works well with Tailwind CSS and provides unstyled, accessible components that you can style as needed. This aligns well with the flexibility and reusability you're aiming for.

Here's a small example of how you could use the `Menu` component from Headless UI with Tailwind in your React project:

```jsx
import { Menu } from '@headlessui/react'

const Example = () => (
  <Menu>
    <Menu.Button className="bg-blue-500 text-white px-4 py-2 rounded">
      Options
    </Menu.Button>
    <Menu.Items className="mt-2 bg-white border border-gray-300 rounded shadow-lg">
      <Menu.Item>
        {({ active }) => (
          <a
            href="#"
            className={`block px-4 py-2 ${active ? 'bg-blue-500 text-white' : 'text-gray-900'}`}
          >
            Account settings
          </a>
        )}
      </Menu.Item>
      <Menu.Item>
        {({ active }) => (
          <a
            href="#"
            classN

The model tailors its answer (React/Tailwind ecosystem, no semicolons) even though this session started from zero — the facts crossed the session boundary through the store.

## 7. Episodic memory: remembering past sessions

Semantic memory stores *facts*; episodic memory stores *events* — "what happened, when, and how it went." Each episode is a summarized past session with metadata that supports retrieval:

- **summary** — a dense recap of the session
- **timestamp** — for recency scoring
- **importance** — an LLM-assigned 1–10 rating at write time (a "how much will this matter later?" judgment)
- **keywords** — for cheap relevance matching (production systems use embeddings; see §12)

In [11]:
class EpisodeRecord(BaseModel):
    summary: str
    importance: int      # 1 (trivial) .. 10 (critical to remember)
    keywords: list[str]


class EpisodicMemory:
    """Stores summarized past sessions; retrieves by relevance x recency x importance."""

    def __init__(self, path: str = "./memory_data/episodes.json"):
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.episodes: list[dict] = json.loads(self.path.read_text()) if self.path.exists() else []

    def archive_session(self, messages: list[dict]):
        """Summarize a finished session into an episode (the write policy)."""
        transcript = "\n".join(
            f"{m['role'].upper()}: {m['content']}" for m in messages if isinstance(m.get("content"), str)
        )
        ep = parse_llm(
            transcript,
            system=(
                "Summarize this session for an agent's episodic memory. "
                "Capture what was worked on, decisions, outcomes, and unresolved items. "
                "Rate importance 1-10: how likely is this session to matter in future sessions?"
            ),
            schema=EpisodeRecord,
        )
        self.episodes.append({
            "summary": ep.summary,
            "importance": ep.importance,
            "keywords": [k.lower() for k in ep.keywords],
            "timestamp": time.time(),
        })
        self.path.write_text(json.dumps(self.episodes, indent=2))
        return ep


episodic = EpisodicMemory()
episode = episodic.archive_session(chat.messages)
print(f"importance={episode.importance}  keywords={episode.keywords}")
print(textwrap.fill(episode.summary, 100))

importance=7  keywords=['Aadhil', 'Python', 'favorite language', 'memory recall']
The user, Aadhil, mentioned that their favorite programming language is Python. The assistant
acknowledged this and asked if Aadhil had any specific projects or further interests in Python.
Aadhil then tested the assistant's recall on their name and favorite language, which the assistant
correctly remembered.


## 8. Procedural memory: learning *how* to behave

Procedural memory is the agent's learned behavior — rules distilled from feedback and experience, injected into every future system prompt. This is how an agent stops repeating mistakes:

> User: "Stop giving me three options every time — just pick one." → rule: *"When choices are roughly equivalent, recommend one option decisively instead of listing alternatives."*

The store is a rule list; the interesting part is the **update step**, where an LLM reconciles new feedback with existing rules (merging, revising, or deleting rather than blindly appending):

In [12]:
class RuleUpdate(BaseModel):
    rules: list[str]        # the complete NEW rule list (replaces the old one)
    reasoning: str


class ProceduralMemory:
    """Learned behavioral rules, updated from user feedback via LLM reconciliation."""

    def __init__(self, path: str = "./memory_data/rules.json"):
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.rules: list[str] = json.loads(self.path.read_text()) if self.path.exists() else []

    def learn(self, feedback: str) -> str:
        update = parse_llm(
            f"Current rules:\n{json.dumps(self.rules, indent=2)}\n\nNew feedback:\n{feedback}",
            system=(
                "You maintain an AI assistant's behavioral rule list. "
                "Given the current rules and new user feedback, return the complete updated list: "
                "merge related rules, revise contradicted ones, keep each rule one imperative sentence, "
                "and keep the list under 10 rules. Only encode durable behavioral preferences, "
                "not one-off requests."
            ),
            schema=RuleUpdate,
        )
        self.rules = update.rules
        self.path.write_text(json.dumps(self.rules, indent=2))
        return update.reasoning

    def as_system_block(self) -> str:
        if not self.rules:
            return ""
        return "Behavioral rules learned from this user:\n" + "\n".join(f"- {r}" for r in self.rules)


procedural = ProceduralMemory()
print(procedural.learn("Please stop giving me three options every time. Just recommend one thing."))
print(procedural.learn("Also, keep answers short — I hate walls of text."))
print()
print(procedural.as_system_block())

The current rule list is empty, and the new user feedback provides a clear guideline to be implemented: to simplify decision-making by offering one recommendation instead of multiple options.
The new feedback suggests a preference for brevity in communication, aligning with the existing rule to simplify suggestions by limiting them to one option. By combining these insights, we maintain a concise and direct style across all interactions without conflicting with the current rule.

Behavioral rules learned from this user:
- Recommend only one option when providing suggestions.
- Keep answers concise and avoid lengthy responses.


## 9. Retrieval scoring: which memories make the cut?

You can't inject everything — context is finite and irrelevant memories actively *hurt* (they distract the model). The classic scoring function comes from the *Generative Agents* paper (Park et al., 2023):

$$score = w_{rel} \cdot relevance + w_{rec} \cdot recency + w_{imp} \cdot importance$$

- **relevance** — similarity between the memory and the current query (embeddings in production; keyword overlap here)
- **recency** — exponential decay by age (newer memories score higher)
- **importance** — the write-time 1–10 rating


In [13]:
def score_episode(episode: dict, query: str, *, half_life_days: float = 7.0,
                  w_rel: float = 1.0, w_rec: float = 0.5, w_imp: float = 0.3) -> float:
    # Relevance: fraction of query words that hit the episode's keywords/summary (0..1)
    q_words = set(query.lower().split())
    ep_words = set(episode["keywords"]) | set(episode["summary"].lower().split())
    relevance = len(q_words & ep_words) / max(len(q_words), 1)

    # Recency: exponential decay with a configurable half-life (0..1)
    age_days = (time.time() - episode["timestamp"]) / 86400
    recency = math.exp(-math.log(2) * age_days / half_life_days)

    # Importance: normalize the 1-10 rating (0..1)
    importance = episode["importance"] / 10

    return w_rel * relevance + w_rec * recency + w_imp * importance


def retrieve_episodes(memory: EpisodicMemory, query: str, top_k: int = 3) -> list[dict]:
    ranked = sorted(memory.episodes, key=lambda e: score_episode(e, query), reverse=True)
    return ranked[:top_k]


# Seed a couple of synthetic older episodes so retrieval has something to rank
episodic.episodes += [
    {"summary": "Debugged a CORS error in the legal CRM's API layer; fixed by adding middleware.",
     "importance": 7, "keywords": ["cors", "crm", "api", "bug"], "timestamp": time.time() - 3 * 86400},
    {"summary": "Discussed vacation plans for December; user is traveling to Japan.",
     "importance": 3, "keywords": ["vacation", "japan", "travel"], "timestamp": time.time() - 1 * 86400},
]

query = "I'm seeing another API bug in the CRM"
for ep in retrieve_episodes(episodic, query):
    print(f"score={score_episode(ep, query):.3f}  imp={ep['importance']}  {ep['summary'][:80]}")

score=1.206  imp=7  Debugged a CORS error in the legal CRM's API layer; fixed by adding middleware.
score=0.960  imp=7  The user, Aadhil, mentioned that their favorite programming language is Python. 
score=0.543  imp=3  Discussed vacation plans for December; user is traveling to Japan.


The CORS episode outranks the (more recent, but irrelevant) vacation episode — relevance and importance beat pure recency. Tune the weights to your domain: a personal assistant weights recency higher; a technical copilot weights relevance.

## 10. Agent-driven memory: a memory tool via function calling

Everything so far was **application-driven** — *we* decided what to extract and when. The complementary pattern is **agent-driven** memory: give the model a tool and let *it* decide what's worth writing down.

We expose a file-based memory as a single **function tool** with six commands — `view`, `create`, `str_replace`, `insert`, `delete`, `rename` — operating on a `/memories` directory. The model issues commands; our backend executes them.

> **Security note:** every `path` the model sends is untrusted output. Resolve it to canonical form and verify it stays inside your memory root — reject `..`, symlinks, and absolute escapes.

In [14]:
class MemoryToolBackend:
    """Filesystem backend for the agent's memory tool."""

    def __init__(self, root: str = "./memory_data/memories_fs"):
        self.root = Path(root).resolve()
        (self.root / "memories").mkdir(parents=True, exist_ok=True)

    def _resolve(self, path: str) -> Path:
        """Map a model-supplied /memories/... path to disk, rejecting escapes."""
        candidate = (self.root / path.lstrip("/")).resolve()
        if not candidate.is_relative_to(self.root):
            raise ValueError(f"Path escapes memory root: {path}")
        return candidate

    def execute(self, tool_input: dict) -> str:
        cmd = tool_input["command"]
        if cmd == "view":
            p = self._resolve(tool_input["path"])
            if p.is_dir():
                entries = sorted(str(e.relative_to(self.root)) for e in p.rglob("*"))
                return "Directory: /" + "\n/".join(entries) if entries else "(empty directory)"
            lines = p.read_text().splitlines()
            return "\n".join(f"{i + 1}: {line}" for i, line in enumerate(lines))
        if cmd == "create":
            p = self._resolve(tool_input["path"])
            p.parent.mkdir(parents=True, exist_ok=True)
            p.write_text(tool_input["file_text"])
            return f"Created {tool_input['path']}"
        if cmd == "str_replace":
            p = self._resolve(tool_input["path"])
            content = p.read_text()
            if content.count(tool_input["old_str"]) != 1:
                return "Error: old_str must match exactly once"
            p.write_text(content.replace(tool_input["old_str"], tool_input["new_str"]))
            return f"Edited {tool_input['path']}"
        if cmd == "insert":
            p = self._resolve(tool_input["path"])
            lines = p.read_text().splitlines()
            lines.insert(tool_input["insert_line"], tool_input["insert_text"].rstrip("\n"))
            p.write_text("\n".join(lines) + "\n")
            return f"Inserted into {tool_input['path']}"
        if cmd == "delete":
            p = self._resolve(tool_input["path"])
            if p.is_file():
                p.unlink()
            return f"Deleted {tool_input['path']}"
        if cmd == "rename":
            src = self._resolve(tool_input["old_path"])
            dst = self._resolve(tool_input["new_path"])
            dst.parent.mkdir(parents=True, exist_ok=True)
            src.rename(dst)
            return f"Renamed to {tool_input['new_path']}"
        return f"Error: unknown command {cmd!r}"


memory_backend = MemoryToolBackend()
print("Memory root:", memory_backend.root)

Memory root: /Users/aadhilimam/Desktop/my_code/articles/Agentic Memory/memory_data/memories_fs


In [15]:
MEMORY_TOOL = {
    "type": "function",
    "function": {
        "name": "memory",
        "description": (
            "Read and write your persistent memory directory (/memories). "
            "Use 'view' to list the directory or read a file, 'create' to write a file, "
            "'str_replace' to edit, 'insert' to add a line, 'delete' and 'rename' to manage files. "
            "Check /memories at the start of a task; record durable facts as you learn them."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "command": {"type": "string",
                            "enum": ["view", "create", "str_replace", "insert", "delete", "rename"]},
                "path": {"type": "string", "description": "Path under /memories, e.g. /memories/user.md"},
                "file_text": {"type": "string", "description": "Full file contents (for create)"},
                "old_str": {"type": "string", "description": "Exact text to replace (for str_replace)"},
                "new_str": {"type": "string", "description": "Replacement text (for str_replace)"},
                "insert_line": {"type": "integer", "description": "Line number to insert after (for insert)"},
                "insert_text": {"type": "string", "description": "Text to insert (for insert)"},
                "old_path": {"type": "string", "description": "Source path (for rename)"},
                "new_path": {"type": "string", "description": "Destination path (for rename)"},
            },
            "required": ["command"],
        },
    },
}


def run_agent_with_memory_tool(user_message: str, max_iterations: int = 10) -> str:
    """A manual agentic loop: the model reads/writes its /memories directory as it works."""
    messages = [
        {"role": "system", "content": (
            "You are a personal assistant with a persistent memory directory. "
            "At the start of a task, check /memories for relevant notes. "
            "When you learn something durable about the user, record it in a memory file."
        )},
        {"role": "user", "content": user_message},
    ]

    for _ in range(max_iterations):
        response = client.chat.completions.create(
            model=MODEL,
            max_completion_tokens=2048,
            tools=[MEMORY_TOOL],
            messages=messages,
        )
        msg = response.choices[0].message

        if not msg.tool_calls:
            return msg.content

        # Echo the assistant turn, execute every tool call, return one result per call
        messages.append(msg)
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            print(f"  [memory] {args.get('command')} {args.get('path', '')}")
            try:
                result = memory_backend.execute(args)
            except Exception as exc:
                result = f"Error: {exc}"
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

    return "(hit iteration limit)"


# Session A: the agent decides to write things down
print(run_agent_with_memory_tool(
    "Remember these for later: I take my coffee black, my standup is at 9:30 IST, "
    "and my current sprint goal is shipping the invoice module."
))

  [memory] create /memories/coffee_preference.md
  [memory] create /memories/work_schedule.md
  [memory] create /memories/sprint_goal.md
Got it! I've saved the following information for you:

- You take your coffee black.
- Your standup is at 9:30 IST.
- Your current sprint goal is shipping the invoice module.

Let me know if there's anything else you'd like to add or update!


In [16]:
# Session B: a completely fresh conversation — the agent recovers state from its own files
print(run_agent_with_memory_tool("What's my sprint goal, and when is my standup?"))

  [memory] view /memories/sprint_goals.md
  [memory] view /memories/standup_time.md
I don't have any information about your sprint goals or standup times saved. Could you let me know what they are so I can remember them for future reference?


In [17]:
# Peek at what the agent actually wrote to disk
for f in sorted(memory_backend.root.rglob("*")):
    if f.is_file():
        print(f"── {f.relative_to(memory_backend.root)} " + "─" * 40)
        print(f.read_text())

── memories/coffee_preference.md ────────────────────────────────────────
User takes their coffee black.
── memories/sprint_goal.md ────────────────────────────────────────
User's current sprint goal is shipping the invoice module.
── memories/work_schedule.md ────────────────────────────────────────
User's standup is at 9:30 IST.


**Application-driven vs. agent-driven — when to use which?**

| | Application-driven (§6–8) | Agent-driven (memory tool) |
|---|---|---|
| Write decisions | Deterministic pipeline | Model's judgment |
| Consistency | High — same extraction every time | Variable — depends on prompting |
| Flexibility | Only captures what you coded for | Captures anything the model deems useful |
| Cost | Extra LLM calls per session | Tool-call tokens inside the session |
| Best for | User profiles, compliance, analytics | Open-ended assistants, coding agents, scratchpads |

## 11. Putting it all together: the `MemoryAgent`

Now we compose every layer into one agent that implements the §3 architecture:

- **Retrieve**: semantic facts + top-scored episodes + procedural rules → system prompt
- **Working memory**: the running `messages` array
- **Consolidate**: on `end_session()`, archive the episode and extract new facts

In [18]:
class MemoryAgent:
    """An agent wiring together working, semantic, episodic, and procedural memory."""

    def __init__(self, persona: str = "You are a helpful personal assistant."):
        self.persona = persona
        self.semantic = SemanticMemory()
        self.episodic = EpisodicMemory()
        self.procedural = ProceduralMemory()
        self.messages: list[dict] = []          # working memory

    # ---- 1. RETRIEVE + 2. ASSEMBLE -------------------------------------
    def _build_system(self, query: str) -> str:
        parts = [self.persona]
        if block := self.procedural.as_system_block():
            parts.append(block)
        if block := self.semantic.as_system_block(query):
            parts.append(block)
        episodes = retrieve_episodes(self.episodic, query, top_k=2)
        if episodes:
            parts.append("Relevant past sessions:\n" + "\n".join(f"- {e['summary']}" for e in episodes))
        return "\n\n".join(parts)

    # ---- 3. RESPOND ------------------------------------------------------
    def chat(self, user_message: str, max_tokens: int = 1024) -> str:
        self.messages.append({"role": "user", "content": user_message})

        # Keep working memory bounded (client-side compaction from §5)
        if len(self.messages) > 20:
            self.messages = compact(self.messages, keep_recent=8)

        reply = llm(self.messages, system=self._build_system(user_message), max_tokens=max_tokens)
        self.messages.append({"role": "assistant", "content": reply})
        return reply

    def give_feedback(self, feedback: str) -> str:
        """Route explicit feedback into procedural memory."""
        return self.procedural.learn(feedback)

    # ---- 4. CONSOLIDATE --------------------------------------------------
    def end_session(self):
        if not self.messages:
            return
        transcript = "\n".join(
            f"{m['role'].upper()}: {m['content']}" for m in self.messages if isinstance(m.get("content"), str)
        )
        self.semantic.add(extract_facts(transcript))   # facts -> semantic store
        self.episodic.archive_session(self.messages)   # session -> episodic store
        self.messages = []                             # clear working memory
        print("[session consolidated: facts extracted, episode archived]")

In [19]:
agent = MemoryAgent()

# ---- Session 1 ----
print("SESSION 1")
print(agent.chat("I'm adding invoice PDF export to my CRM. Which library should I use?"))
agent.end_session()

SESSION 1
I recommend using `pdf-lib` for generating PDF invoices in your CRM. It's a versatile and well-documented library that works well with JavaScript and should integrate smoothly with your React project.
[session consolidated: facts extracted, episode archived]


In [20]:
# ---- Session 2 (fresh working memory; long-term stores carry over) ----
print("SESSION 2")
print(agent.chat("Following up on the export feature we discussed — what was your recommendation again?"))
agent.end_session()

SESSION 2
I recommended using the `pdf-lib` library to implement the export feature for invoices as PDFs in your CRM project.
[session consolidated: facts extracted, episode archived]


Session 2 starts with an empty `messages` array, yet the agent recalls the PDF-export discussion — retrieved from the episodic store, plus user facts from the semantic store, plus behavioral rules from procedural memory (notice the answers are decisive and short, per §8's learned rules).

## 12. Advanced topics

### 12.1 Forgetting and decay
Memory that only grows becomes noise. Production systems prune:
- **TTL / decay** — drop or down-weight memories below a score threshold (the §9 recency term already implements soft decay).
- **Supersession** — when a new fact contradicts an old one ("user moved to Berlin"), *replace*, don't append. The §8 LLM-reconciliation pattern generalizes to semantic facts.
- **User-driven deletion** — "forget what I said about X" must actually delete (and matters for GDPR/CCPA compliance).

### 12.2 Consolidation and reflection
Periodically run a background pass over raw memories to merge duplicates, abstract patterns ("user asked about testing 5 times → user cares deeply about test coverage"), and fix stale facts. This mirrors how human memory consolidates during sleep, and it's exactly what §8's `learn()` does for rules — schedule the same reconciliation for facts and episodes.

### 12.3 Vector stores and embeddings
Our keyword-overlap relevance is deliberately dependency-free. In production, replace it with embedding similarity using OpenAI's embeddings endpoint:

```python
def embed(text: str) -> list[float]:
    return client.embeddings.create(model="text-embedding-3-small", input=text).data[0].embedding
```

Embed each memory at write time, embed the query at read time, and rank by cosine similarity (FAISS/Chroma/pgvector at scale). OpenAI also offers hosted **vector stores** with the `file_search` tool if you'd rather not run retrieval yourself. Everything else — the scoring formula, the write policies, the architecture — stays identical; only the `relevance` term changes.

### 12.4 Memory and prompt caching
OpenAI applies **automatic prompt caching** to prompts over ~1024 tokens: cached prefixes are billed at a discount and served faster. Caching is a **prefix match**, so memory injection strategy matters:
- Keep the stable persona + rules **first** in the system prompt.
- Inject volatile per-query recalled memories **after** the stable prefix (or into the user turn) so they don't break prefix reuse.
- Never interpolate timestamps at the top of the system prompt.

### 12.5 Hosted state and memory services
The **Responses API** (§5.4) gives you hosted *working* memory (`store=True` + `previous_response_id`). For hosted *long-term* retrieval, pair it with OpenAI vector stores + `file_search`. Third-party memory layers (e.g. mem0, Zep, LangGraph's memory store) package the extract-store-retrieve pipeline from §6–9 as a service — under the hood they implement exactly the architecture in §3.

### 12.6 Multi-user isolation
One store per user, always. Our file backend has no access control — in multi-tenant systems, key every store by authenticated user ID and never let one user's memories leak into another's prompt.

## 13. Best practices, pitfalls, and where to go next

### Best practices
1. **Design the write policy first.** "Store everything, filter at read time" drowns retrieval in noise. Be selective at write time (importance ratings, extraction prompts that skip trivia).
2. **Atomic facts beat blobs.** One fact per record makes dedup, supersession, and deletion tractable.
3. **Layer your memories.** Working → episodic → semantic → procedural each solve a different problem; most real agents need at least three of the four.
4. **Score retrieval on relevance × recency × importance,** and tune the weights per domain.
5. **Consolidate on a schedule.** Reflection passes keep the store small and coherent.
6. **Validate every model-supplied path** in memory-tool backends (canonicalize, then check containment).
7. **Never store secrets** (API keys, passwords, tokens) in memory files, and mind PII regulations before persisting user data.

### Common pitfalls
- **Context stuffing** — injecting 50 memories "just in case" degrades answers and costs tokens. Top-k with a score floor.
- **Stale facts** — appending without reconciling means the prompt eventually contains both "lives in Colombo" and "lives in Berlin."
- **Cache-busting injection** — putting volatile memories at the top of the prompt breaks OpenAI's automatic prefix caching.
- **Wrong tokenizer encoding** — use `tiktoken.encoding_for_model()` (GPT-4o+ is `o200k_base`, not `cl100k_base`), and remember counts are estimates due to chat-format overhead.
- **Memory as a crutch for bad prompts** — if the agent needs the same instruction every session, that's a system-prompt fix, not a memory.

### References & further reading
- Park et al., *Generative Agents: Interactive Simulacra of Human Behavior* (2023) — the retrieval-scoring formula and reflection
- Packer et al., *MemGPT: Towards LLMs as Operating Systems* (2023) — paged/tiered memory management
- OpenAI docs: **Function calling**, **Structured outputs**, **Responses API & conversation state**, **Prompt caching**, **Embeddings & vector stores** — https://platform.openai.com/docs

---
*You now have a complete, working agentic memory stack: run the cells, inspect `./memory_data/`, and swap the JSON stores for your database of choice when you're ready for production.*